In [ ]:
# --- ⏱ Start total runtime timer ---
import time
start_time = time.time()
print("🚀 Execution started...")


!pip install --quiet dask[complete] pyarrow fastparquet scikit-learn scipy matplotlib tqdm joblib


import os, time, shutil, joblib, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import dask.dataframe as dd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Lasso, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_squared_error, accuracy_score, f1_score
from scipy.optimize import minimize

# === Change this to your file path ===
DATA_PATH = "/content/drive/MyDrive/stocks_merged_clean.csv"
OUT_PARQUET = "/content/drive/MyDrive/stocks_processed_parquet_final/"
print("Input:", DATA_PATH)


# Batch processing and save per-ticker parquet parts (memory-safe)
if os.path.exists(OUT_PARQUET):
    shutil.rmtree(OUT_PARQUET)
os.makedirs(OUT_PARQUET, exist_ok=True)

# Define data types for relevant columns to avoid TypeError
dtype = {
    "Close": "float64",
    "PrevClose": "float64",
    "DailyReturn": "float64",
    "FutureReturn": "float64",
    "LabelUp": "int8",
    "Volume": "int64",
    "SentimentScore": "float64",
    "Sentiment": "object", # Keep Sentiment as object for now
    "Ticker": "object" # Keep Ticker as object for now
}

chunks = pd.read_csv(DATA_PATH, chunksize=200000, parse_dates=['Date'], dtype=dtype)
processed_parts = []
for i, chunk in enumerate(tqdm(chunks, desc="Processing chunks")):
    chunk = chunk.sort_values(["Ticker","Date"]).reset_index(drop=True)
    # compute prev_close and returns
    chunk["PrevClose"] = chunk.groupby("Ticker")["Close"].shift(1)
    chunk["DailyReturn"] = (chunk["Close"] - chunk["PrevClose"]) / chunk["PrevClose"]
    chunk = chunk.dropna(subset=["DailyReturn"]).reset_index(drop=True)
    chunk["FutureReturn"] = chunk.groupby("Ticker")["DailyReturn"].shift(-1)
    chunk = chunk.dropna(subset=["FutureReturn"]).reset_index(drop=True)
    chunk["LabelUp"] = (chunk["FutureReturn"] > 0).astype("int8")
    part_path = os.path.join(OUT_PARQUET, f"part_{i:03}.parquet")
    chunk.to_parquet(part_path, index=False)
    processed_parts.append(part_path)

print("✅ Preprocessed batches saved at:", OUT_PARQUET)


import dask.dataframe as dd
import os

# Define the path where your part_*.parquet files are saved
OUT_PARQUET = "/content/drive/MyDrive/stocks_processed_parquet_final/"

# Read all parquet parts into one Dask DataFrame
ddf = dd.read_parquet(os.path.join(OUT_PARQUET, "*.parquet"))
print("✅ Dask DataFrame loaded successfully with", ddf.npartitions, "partitions")
print("Columns:", list(ddf.columns))


import pyarrow as pa
import dask.dataframe as dd
import shutil, os

# Define output path again
combined_path = OUT_PARQUET + "combined_fixed/"

# Remove old folder if exists
if os.path.exists(combined_path):
    shutil.rmtree(combined_path)

# Force sentiment column to string (across all partitions)
ddf["Sentiment"] = ddf["Sentiment"].astype(str)

# Optional safety: also ensure correct types for other columns
ddf["Volume"] = ddf["Volume"].astype("int64")
ddf["SentimentScore"] = ddf["SentimentScore"].astype("float64")
ddf["Close"] = ddf["Close"].astype("float64")
ddf["High"] = ddf["High"].astype("float64")
ddf["Low"] = ddf["Low"].astype("float64")
ddf["Open"] = ddf["Open"].astype("float64")
ddf["PrevClose"] = ddf["PrevClose"].astype("float64")
ddf["DailyReturn"] = ddf["DailyReturn"].astype("float64")
ddf["FutureReturn"] = ddf["FutureReturn"].astype("float64")
ddf["LabelUp"] = ddf["LabelUp"].astype("int8")

# ✅ Define explicit schema for parquet writer
schema = pa.schema([
    ("Ticker", pa.string()),
    ("Date", pa.timestamp("ns")),
    ("Close", pa.float64()),
    ("High", pa.float64()),
    ("Low", pa.float64()),
    ("Open", pa.float64()),
    ("Volume", pa.int64()),
    ("Sentiment", pa.string()),
    ("SentimentScore", pa.float64()),
    ("PrevClose", pa.float64()),
    ("DailyReturn", pa.float64()),
    ("FutureReturn", pa.float64()),
    ("LabelUp", pa.int8())
])

# Write parquet using explicit schema
print("Writing parquet with enforced schema...")
ddf.to_parquet(
    combined_path,
    engine="pyarrow",
    write_index=False,
    schema=schema,
    overwrite=True
)

print("✅ Combined parquet written successfully to:", combined_path)